# Financials Sector Portfolio Optimization

**Analyst:** Anurag Pokala  
**Date:** February 5, 2026

## Overview

This notebook implements three portfolio optimization techniques for the Financials sector sleeve:
1. **Mean-Variance (Markowitz) Optimization**
2. **Black-Litterman with Macro Views (Interest Rate Forecasts)**
3. **CVaR (Conditional Value-at-Risk) Optimization**

**Note:** Unlike other sectors, this Financials sector optimization uses a **macro-driven Black-Litterman** approach based on interest rate forecasts and sector-specific rate sensitivities, rather than sentiment analysis. This is particularly appropriate for Financials given their high sensitivity to interest rate changes.

## Section 1: Setup and Configuration

In [ ]:
# Import libraries
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
import warnings
warnings.filterwarnings('ignore')

# Auto-reload modules when they change
%load_ext autoreload
%autoreload 2

# Add src to path
sys.path.insert(0, os.path.abspath('..'))

# Import custom modules
from src import data_loader, estimators, constraints, mv_optimizer, bl_macro, metrics, reporting, backtester, cvar_optimizer

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

print("✓ All libraries imported successfully")

In [ ]:
# Load Financials sector configuration
with open('../config_financials.yaml', 'r') as f:
    config = yaml.safe_load(f)

# Extract key parameters
tickers = config['tickers']
before_weights = np.array([config['before_weights'][t] for t in tickers])
start_date = config['data']['start_date']
end_date = config['data']['end_date']
rf = config['risk_free_rate']

print("Financials Sector Configuration Loaded:")
print(f"  Tickers: {', '.join(tickers)}")
print(f"  Date range: {start_date} to {end_date}")
print(f"  Risk-free rate: {rf:.2%}")
print(f"  Portfolio value: ${config['portfolio']['total_value']:,}")
print(f"\nBefore Portfolio Weights (Current Portfolio as Prior):")
for ticker, weight in zip(tickers, before_weights):
    print(f"  {ticker}: {weight:.2%}")
print(f"  Total: {before_weights.sum():.2%}")
print(f"\nConstraints:")
print(f"  Long-only: {config['constraints']['long_only']}")
print(f"  Fully invested: {config['constraints']['fully_invested']}")
print(f"  Max weight: {config['constraints']['max_weight']:.0%}")

## Section 2: Data Loading and Preprocessing

Load 3 years of daily price data for Financials sector stocks.

In [ ]:
# Load price data
prices = data_loader.load_prices(tickers, start_date, end_date)

# Display summary
print("\nFirst 5 days:")
print(prices.head())
print("\nLast 5 days:")
print(prices.tail())

In [ ]:
# Calculate simple returns (required for backtest consistency)
returns = data_loader.compute_returns(prices, method='simple')

print("\nReturns Summary (simple returns):")
print(f"Shape: {returns.shape}")
print(f"Date range: {returns.index[0]} to {returns.index[-1]}")
print(f"\nAnnualized Statistics:")
print(f"{'Ticker':<8} {'Mean Return':<15} {'Volatility':<15} {'Sharpe'}")
print("-" * 55)
for ticker in tickers:
    mean_ret = returns[ticker].mean() * 252
    vol = returns[ticker].std() * np.sqrt(252)
    sharpe = (mean_ret - rf) / vol if vol > 0 else 0
    print(f"{ticker:<8} {mean_ret:>13.2%}  {vol:>13.2%}  {sharpe:>8.3f}")

## Section 3: Covariance and Correlation Analysis

In [ ]:
# Estimate covariance matrix using Ledoit-Wolf shrinkage
Sigma, shrinkage_intensity = estimators.estimate_covariance_shrinkage(returns, method='ledoit_wolf')

print("\nCovariance Matrix (Ledoit-Wolf):")
print(f"Shape: {Sigma.shape}")
print(f"\nAnnualized Volatilities:")
vols = np.sqrt(np.diag(Sigma))
for ticker, vol in zip(tickers, vols):
    print(f"  {ticker}: {vol:.2%}")

In [ ]:
# Calculate correlation matrix
corr_matrix = returns.corr()

# Plot correlation matrix heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
            center=0, vmin=-1, vmax=1, square=True,
            cbar_kws={'label': 'Correlation'})
plt.title('Correlation Matrix - Financials Sector', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('outputs_financials/correlation_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nCorrelation Summary:")
# Extract upper triangle (excluding diagonal)
corr_array = corr_matrix.values
upper_triangle_indices = np.triu_indices_from(corr_array, k=1)
corr_flat = corr_array[upper_triangle_indices]
print(f"  Mean correlation: {corr_flat.mean():.3f}")
print(f"  Max correlation: {corr_flat.max():.3f}")
print(f"  Min correlation: {corr_flat.min():.3f}")

## Section 4: Mean-Variance (Markowitz) Optimization

Classic portfolio optimization maximizing Sharpe ratio.

In [ ]:
# Estimate expected returns (historical mean)
mu_hist = estimators.estimate_expected_returns(returns, shrinkage=0.0)

print("Expected Returns (annualized):")
print("="*50)
for ticker, ret in zip(tickers, mu_hist):
    print(f"{ticker:6s}: {ret:7.2%}")
print("="*50)

In [ ]:
# Run MV optimization
mv_results = mv_optimizer.optimize_mean_variance(
    mu=mu_hist.values,
    Sigma=Sigma.values,
    constraints_config=config['constraints'],
    rf=rf,
    lambda_grid=np.logspace(
        np.log10(config['mean_variance']['lambda_min']),
        np.log10(config['mean_variance']['lambda_max']),
        config['mean_variance']['lambda_points']
    ),
    w_before=before_weights
)

In [ ]:
# Extract optimal weights
mv_weights = mv_results['weights']

print("\n" + "="*70)
print("MEAN-VARIANCE OPTIMIZATION RESULTS")
print("="*70)
print(f"{'Ticker':<8} {'Before':<12} {'MV':<12} {'Change'}")
print("="*70)
for i, ticker in enumerate(tickers):
    print(f"{ticker:<8} {before_weights[i]:>10.2%}  {mv_weights[i]:>10.2%}  {mv_weights[i]-before_weights[i]:>+10.2%}")
print("="*70)
print(f"{'Total':<8} {before_weights.sum():>10.2%}  {mv_weights.sum():>10.2%}")
print("="*70)

# Calculate performance metrics
mv_return = mv_results['return']
mv_vol = mv_results['volatility']
mv_sharpe = mv_results['sharpe']

print(f"\nPortfolio Metrics:")
print(f"  Expected Return: {mv_return:.2%}")
print(f"  Volatility: {mv_vol:.2%}")
print(f"  Sharpe Ratio: {mv_sharpe:.3f}")

## Section 5: Black-Litterman with Macro (Interest Rate) Views

### Primer: Macro-Driven Black-Litterman for Financials

The **Black-Litterman model** is particularly well-suited for the Financials sector when driven by **macroeconomic views** rather than sentiment analysis. Here's why:

#### Why Macro Views for Financials?

1. **High Interest Rate Sensitivity**: Financial stocks have direct exposure to interest rates:
   - **REITs** (O, PLD): High inverse sensitivity (rates ↓ → stock prices ↑)
   - **Capital Markets** (BX, MS, AMP): Moderate sensitivity
   - **Payments** (V): Lower sensitivity
   - **Insurers** (MKL): Positive sensitivity (benefit from float income)

2. **Quantifiable Rate Betas**: Unlike sentiment, rate sensitivity can be estimated from:
   - Historical regression analysis (stock returns vs. rate changes)
   - Duration analysis (for REITs and rate-sensitive stocks)
   - Sector-specific heuristics

3. **Forward-Looking Rate Forecasts**: Central bank policy is more predictable than news sentiment:
   - Fed dot plots and forward guidance
   - Consensus economist forecasts
   - Market-implied rates (futures, swaps)

#### Our Implementation

We use **PyPortfolioOpt's BlackLittermanModel** with:
- **Prior**: Current portfolio weights (not market cap weights)
- **Views**: Generated from rate betas × rate forecast
- **Scenario**: Goldman Sachs consensus (50bps cut, 60% confidence)

**View Formula**:
```
Expected_Return = rate_change_forecast × rate_beta
```

**Example** (50bps cut scenario):
- O (REIT, beta=-15): -0.50 × -15 = +7.5% expected return ✓ Bullish
- MKL (Insurer, beta=+2): -0.50 × +2 = -1.0% expected return ✗ Bearish

This approach tilts the portfolio toward rate-sensitive beneficiaries (REITs) when rate cuts are expected.

In [ ]:
# Extract BL macro parameters
bl_config = config['black_litterman_macro']
rate_forecast = bl_config['scenario']['rate_change_forecast']
confidence = bl_config['scenario']['confidence']
delta = bl_config['delta']
tau = bl_config['tau']

print("Black-Litterman Macro Configuration:")
print(f"  Scenario: {bl_config['scenario']['name']}")
print(f"  Rate forecast: {rate_forecast:+.2f}% ({abs(rate_forecast*100):.0f}bps {'cut' if rate_forecast < 0 else 'hike'})")
print(f"  Confidence: {confidence:.0%}")
print(f"  Risk aversion (delta): {delta}")
print(f"  Uncertainty (tau): {tau}")

In [ ]:
# Run BL macro optimization
bl_results = bl_macro.optimize_bl_macro(
    returns=returns,
    current_weights=before_weights,
    tickers=tickers,
    rate_change_forecast=rate_forecast,
    confidence=confidence,
    constraints_config=config['constraints'],
    delta=delta,
    tau=tau,
    rf=rf,
    Sigma=Sigma
)

# Extract optimal weights
bl_weights = bl_results['weights']

In [ ]:
# Display return comparison
print("\n" + "="*70)
print("RETURN COMPARISON: Prior vs. Posterior")
print("="*70)
print(f"{'Ticker':<8} {'Prior (Equilibrium)':<22} {'Posterior (BL)':<22} {'Change'}")
print("="*70)

for i, ticker in enumerate(tickers):
    prior = bl_results['prior_returns'][i]
    posterior = bl_results['posterior_returns'][i]
    change = posterior - prior
    print(f"{ticker:<8} {prior:>20.2%}  {posterior:>20.2%}  {change:>+10.2%}")

print("="*70)
print("\nInterpretation:")
print("  - Positive changes indicate stocks expected to benefit from rate scenario")
print("  - Larger changes mean stronger macro views vs. equilibrium")
print("="*70)

In [ ]:
# Calculate BL portfolio metrics
bl_return = bl_results['expected_return']
bl_vol = bl_results['volatility']
bl_sharpe = bl_results['sharpe']

print("\nBlack-Litterman Macro Portfolio Metrics:")
print(f"  Expected Return: {bl_return:.2%}")
print(f"  Volatility: {bl_vol:.2%}")
print(f"  Sharpe Ratio: {bl_sharpe:.3f}")

## Section 6: CVaR (Conditional Value-at-Risk) Optimization

### Primer: CVaR for Tail Risk Management

**Conditional Value-at-Risk (CVaR)** minimizes the expected loss in the worst-case scenarios, making it ideal for risk-averse investors.

#### Key Concepts

1. **Value-at-Risk (VaR)**: The maximum loss at a given confidence level
   - Example: 95% VaR = -3% means 5% chance of losing more than 3%

2. **CVaR (Expected Shortfall)**: Average loss **beyond** VaR
   - More conservative than VaR
   - Captures tail risk severity
   - Example: 95% CVaR = -5% means average loss in worst 5% of cases

3. **Why CVaR over Variance?**
   - Variance penalizes upside equally with downside
   - CVaR focuses only on downside (left tail)
   - Better for non-normal return distributions

#### Implementation

We use the **Rockafellar-Uryasev (2000)** Linear Programming formulation:
- Confidence level: 95% (focus on worst 5% of scenarios)
- Constraints: Same as MV (long-only, 20% max weight)
- Uses historical scenario-based optimization

This is particularly relevant for Financials given their exposure to systemic risk (e.g., credit crises, liquidity shocks).

In [ ]:
# Run CVaR optimization
cvar_results = cvar_optimizer.optimize_cvar(
    returns=returns,
    constraints_config=config['constraints'],
    confidence_level=config['cvar']['confidence_level'],
    w_before=before_weights
)

# Extract optimal weights
cvar_weights = cvar_results['weights']

print("\n" + "="*70)
print("CVaR OPTIMIZATION RESULTS")
print("="*70)
print(f"{'Ticker':<8} {'Before':<12} {'CVaR':<12} {'Change'}")
print("="*70)
for i, ticker in enumerate(tickers):
    print(f"{ticker:<8} {before_weights[i]:>10.2%}  {cvar_weights[i]:>10.2%}  {cvar_weights[i]-before_weights[i]:>+10.2%}")
print("="*70)
print(f"{'Total':<8} {before_weights.sum():>10.2%}  {cvar_weights.sum():>10.2%}")
print("="*70)

# Display CVaR metrics
print(f"\nCVaR Portfolio Metrics:")
print(f"  CVaR (95%): {cvar_results['cvar_value']:.2%}")
print(f"  VaR (95%): {cvar_results['var_value']:.2%}")
print(f"  Expected Return: {cvar_results['expected_return']:.2%}")
print(f"  Volatility: {cvar_results['volatility']:.2%}")
print(f"\nInterpretation:")
print(f"  - In the worst 5% of scenarios, expected loss is {abs(cvar_results['cvar_value']):.2%}")
print(f"  - This is an improvement over equal-weight or unconstrained portfolios")

## Section 7: Backtest - Compare All Strategies

Historical performance comparison of:
1. Before (current allocation)
2. Mean-Variance
3. Black-Litterman (Macro)
4. CVaR

In [ ]:
# Prepare portfolio strategies for backtest
portfolios = {
    'Before': before_weights,
    'Mean-Variance': mv_weights,
    'BL (Macro)': bl_weights,
    'CVaR': cvar_weights
}

# Run backtest
backtest_results = {}

for name, weights in portfolios.items():
    results = backtester.backtest_portfolio(
        weights=weights,
        tickers=tickers,
        prices=prices,
        returns=returns,
        risk_free_rate=rf,
        initial_capital=10000
    )
    backtest_results[name] = results
    print(f"✓ Backtested: {name}")

In [ ]:
# Summarize and plot backtest results
summary_df = backtester.summarize_and_plot(
    results_dict=backtest_results,
    rf_annual=rf,
    # save_path='outputs_financials/backtest_comparison.png'
)

print("\n" + "="*100)
print("BACKTEST SUMMARY")
print("="*100)
print(summary_df.to_string())
print("="*100)

In [ ]:
# Display portfolio weights comparison
print("\n" + "="*100)
print("PORTFOLIO WEIGHTS COMPARISON")
print("="*100)

# Create weights DataFrame
weights_df = pd.DataFrame({
    'Ticker': tickers,
    'Before': before_weights,
    'Mean-Variance': mv_weights,
    'BL (Macro)': bl_weights,
    'CVaR': cvar_weights
})

# Format as percentages
weights_display = weights_df.copy()
for col in ['Before', 'Mean-Variance', 'BL (Macro)', 'CVaR']:
    weights_display[col] = weights_display[col].apply(lambda x: f"{x:.2%}")

print(weights_display.to_string(index=False))
print("="*100)

# Calculate and display total turnover for each strategy
print("\nTurnover from Before Portfolio:")
print("-"*100)
mv_turnover = np.sum(np.abs(mv_weights - before_weights))
bl_turnover = np.sum(np.abs(bl_weights - before_weights))
cvar_turnover = np.sum(np.abs(cvar_weights - before_weights))

print(f"  Mean-Variance: {mv_turnover:.2%}")
print(f"  BL (Macro):    {bl_turnover:.2%}")
print(f"  CVaR:          {cvar_turnover:.2%}")
print("="*100)

## Section 8: Final Summary and Interpretation

### Key Findings

This Financials sector optimization demonstrates:

1. **Macro-Driven Black-Litterman**: 
   - Tilts toward rate-sensitive beneficiaries (REITs) given expected rate cuts
   - More transparent and quantifiable than sentiment-based views
   - Suitable for sector-wide macro themes

2. **Mean-Variance**: 
   - Maximizes Sharpe ratio using historical data
   - Risk-return efficient but backward-looking

3. **CVaR**:
   - Minimizes tail risk (worst 5% scenarios)
   - More conservative, suitable for risk-averse mandate

### Implementation Notes

- **Tech Stack**: PyPortfolioOpt for BL (cleaner API for macro views)
- **Rate Betas**: Heuristic estimates (can refine with regression)
- **Scenario Analysis**: Current model uses single scenario; can extend to multiple scenarios with probability weights

### Next Steps

1. Validate rate betas using historical regression
2. Test multiple rate scenarios (cuts, hikes, holds)
3. Consider adding credit spread views for additional Financials exposure
4. Backtest on out-of-sample period when Fed policy changes